In [1]:
import numpy as np   #从文件中导入numpy
import torch     #导入torch   我需要看一下torch在那里里面有哪些需要被使用到的功能
from torch.utils import data   #从torch.utils 中导入， data torch utils 打他tensordataset 
from d2l import torch as d2l  #从d2l中导入torch 并且命名为 d2l 

In [2]:
true_w = torch.tensor([2,-3.4])   #定义一个true_w 以tensor的形式
true_b = 4.2 
features ,labels = d2l.synthetic_data(true_w,true_b,1000)  #这里非常值得看源码 d2l.synthetic_data(...) 从d2l.中调用 synthetic_data 的函数并生成我们要的features和labels 

In [3]:
#读取数据
def load_array (data_array,batch_size,is_train = True):  #加载数组，这里输入的data_array（features,labels)是什么， batch_size是， 批次内的数量， train 是指是否要记录梯度对吗
    """构造一个Pytorch数据迭代器 """   #is tranin = true 问的是训练的时候是否把数据打乱
    dataset = data.TensorDataset(*data_array)  #这里的data_array 是指features,labels 
    return data.DataLoader(dataset,batch_size,shuffle = is_train) #返回，dataloader是一个属性吗， 这返回时返回要给什么东西， 为什么不是返回一个东西
    #

In [17]:
data.DataLoader??

Init signature:
data.DataLoader(
    dataset: 'Dataset[_T_co]',
    batch_size: 'int | None' = 1,
    shuffle: 'bool | None' = None,
    sampler: 'Sampler | Iterable | None' = None,
    batch_sampler: 'Sampler[list] | Iterable[list] | None' = None,
    num_workers: 'int' = 0,
    collate_fn: '_collate_fn_t | None' = None,
    pin_memory: 'bool' = False,
    drop_last: 'bool' = False,
    timeout: 'float' = 0,
    worker_init_fn: '_worker_init_fn_t | None' = None,
    multiprocessing_context=None,
    generator=None,
    *,
    prefetch_factor: 'int | None' = None,
    persistent_workers: 'bool' = False,
    pin_memory_device: 'str' = '',
    in_order: 'bool' = True,
) -> 'None'
Source:        
class DataLoader(Generic[_T_co]):
    r"""
    Data loader combines a dataset and a sampler, and provides an iterable over the given dataset.

    The :class:`~torch.utils.data.DataLoader` supports both map-style and
    iterable-style datasets with single- or multi-process loading, customizing
 

In [4]:
batch_size = 10 
data_iter = load_array((features,labels),batch_size)
print(data_iter)

In [5]:
print(type(data_iter)) #data_iter是一个DataLoader类型的对象，

<class 'torch.utils.data.dataloader.DataLoader'>


In [6]:
print(data_iter.batch_size)

10


In [7]:
print(type(data_iter.dataset))

<class 'torch.utils.data.dataset.TensorDataset'>


In [8]:
next(iter(data_iter))   #这里的next是什么

[tensor([[ 0.0244,  0.1084],
         [ 0.6975,  0.2095],
         [-0.5193, -0.0340],
         [ 0.1134, -0.5808],
         [ 0.1285, -0.8951],
         [-0.6887,  0.8390],
         [-0.1794,  1.4142],
         [-1.0529,  0.7421],
         [-0.2820, -0.2699],
         [-0.0649,  1.0286]]),
 tensor([[ 3.8728],
         [ 4.8814],
         [ 3.2714],
         [ 6.3880],
         [ 7.4992],
         [-0.0201],
         [-0.9873],
         [-0.4416],
         [ 4.5677],
         [ 0.5689]])]

In [9]:
from torch import nn  #从torch导入nn 这个模组
net = nn.Sequential(nn.Linear(2,1))  #这里的nn 是导入到net 说的是什么

In [10]:
net[0].weight.data.normal_(0,0.01)  # net 
net[0].bias.data.fill_(0) #

tensor([0.])

In [11]:
loss =nn.MSELoss()  #nn 里面自带了损失函数

In [12]:
trainer = torch.optim.SGD(net.parameters(),lr=0.03)   #这题是随机梯度下降


In [15]:
num_epochs= 3 #
for epoch in range (num_epochs):  #大循环3次
    for X,y in data_iter:  #从data_iter 中取出每次需要训练的X，y 
        #print("每次拿10个X：",X.shape)
        #print("每次拿10个y：",y.shape)
        l = loss(net(X),y) #求损失函数 算y_hat
        trainer.zero_grad()  # 梯度清零是吗
        l.backward()  #反向求梯度
        trainer.step()  #更新参数
    l = loss(net(features),labels)  # 训练玩着一整个epoch后， 用当前模型，重新对全部的features做一次预测然后计算整个数据集的MSE
    print(f'epoch{epoch+1},loss{1:f}')
    print(f'epoch{epoch+1},loss{l:f}')

epoch1,loss1.000000
epoch1,loss0.000096
epoch2,loss1.000000
epoch2,loss0.000096
epoch3,loss1.000000
epoch3,loss0.000096


In [14]:
w =net[0].weight.data
print("w的估计误差：",true_w - w. reshape(true_w.shape))
b =net[0].bias.data
print("b的估计误差： ",true_b - b)


w的估计误差： tensor([ 0.0003, -0.0002])
b的估计误差：  tensor([0.0006])
